# Feature Engineering

## Tech Challenge Fase 3 - Machine Learning Engineering

**Objetivo:** Preparar os dados para modelagem supervisionada (Classificação e Regressão) através da criação de features relevantes e transformações necessárias.

### Estrutura:
1. Carregamento dos Dados
2. Criação de Features Temporais
3. Criação de Features de Contexto Operacional
4. Encoding de Variáveis Categóricas
5. Criação das Variáveis Target
6. Análise de Correlação das Novas Features
7. Tratamento de Outliers
8. Divisão Train/Test e Salvamento

In [ ]:
# Imports e Configurações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
import joblib
import os

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Cores para gráficos
COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

print('✅ Bibliotecas carregadas com sucesso!')

---
## 1. Carregamento dos Dados

Carregamos os dados já processados na EDA para aplicar as transformações de Feature Engineering.

In [ ]:
# Carrega os dados processados da EDA
df = pd.read_parquet('../data/processed/flights_processed.parquet')
print(f'📊 Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'\n📋 Colunas disponíveis:')
print(df.columns.tolist())

In [ ]:
# Visualiza os primeiros registros
df.head()

In [ ]:
# Verifica tipos de dados
df.dtypes

In [ ]:
# =============================================================================
# FILTRAR VOOS CANCELADOS
# =============================================================================
# Para modelagem de atrasos, removemos voos cancelados

df_model = df[df['CANCELLED'] == 0].copy()
print(f'✅ Voos cancelados removidos')
print(f'   Antes: {len(df):,} voos')
print(f'   Depois: {len(df_model):,} voos')
print(f'   Removidos: {len(df) - len(df_model):,} voos cancelados')

---
## 2. Criação de Features Temporais

Extraímos informações temporais detalhadas para capturar padrões de atrasos ao longo do tempo.

In [ ]:
# =============================================================================
# FEATURES TEMPORAIS
# =============================================================================

# 1. HOUR - Hora do voo programado
df_model['HOUR'] = (df_model['SCHEDULED_DEPARTURE'] // 100).astype(int)
df_model['HOUR'] = df_model['HOUR'].clip(0, 23)  # Garante valores válidos 0-23
print('✅ HOUR criada (hora do voo programado)')

# 2. MINUTE - Minuto do voo programado
df_model['MINUTE'] = (df_model['SCHEDULED_DEPARTURE'] % 100).astype(int)
df_model['MINUTE'] = df_model['MINUTE'].clip(0, 59)
print('✅ MINUTE criada (minuto do voo programado)')

# 3. PERIOD - Período do dia (categórico)
def get_period(hour):
    """Classifica o período do dia baseado na hora."""
    if 5 <= hour < 12:
        return 'Manhã'
    elif 12 <= hour < 18:
        return 'Tarde'
    elif 18 <= hour < 22:
        return 'Noite'
    else:
        return 'Madrugada'

df_model['PERIOD'] = df_model['HOUR'].apply(get_period)
print('✅ PERIOD criada (Manhã/Tarde/Noite/Madrugada)')

# 4. IS_WEEKEND - Fim de semana
df_model['IS_WEEKEND'] = (df_model['DAY_OF_WEEK'] >= 6).astype(int)
print('✅ IS_WEEKEND criada (1=Sáb/Dom, 0=Seg-Sex)')

# 5. IS_MORNING_RUSH - Rush matinal (6h-9h)
df_model['IS_MORNING_RUSH'] = ((df_model['HOUR'] >= 6) & (df_model['HOUR'] <= 9)).astype(int)
print('✅ IS_MORNING_RUSH criada (6h-9h)')

# 6. IS_EVENING_RUSH - Rush vespertino (17h-20h)
df_model['IS_EVENING_RUSH'] = ((df_model['HOUR'] >= 17) & (df_model['HOUR'] <= 20)).astype(int)
print('✅ IS_EVENING_RUSH criada (17h-20h)')

# 7. QUARTER - Trimestre
df_model['QUARTER'] = ((df_model['MONTH'] - 1) // 3 + 1).astype(int)
print('✅ QUARTER criada (1-4)')

# 8. IS_HOLIDAY_SEASON - Temporada de férias (Jun-Ago, Dez)
df_model['IS_HOLIDAY_SEASON'] = df_model['MONTH'].isin([6, 7, 8, 12]).astype(int)
print('✅ IS_HOLIDAY_SEASON criada (Jun-Ago, Dez)')

# Visualiza distribuição do período
print(f'\n📊 Distribuição por PERIOD:')
print(df_model['PERIOD'].value_counts())

In [ ]:
# Visualização das features temporais
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Distribuição por hora
ax1 = axes[0, 0]
df_model['HOUR'].value_counts().sort_index().plot(kind='bar', ax=ax1, color=COLORS[0], alpha=0.7)
ax1.set_title('Distribuição por Hora', fontsize=12, fontweight='bold')
ax1.set_xlabel('Hora')
ax1.set_ylabel('Frequência')
ax1.tick_params(axis='x', rotation=0)

# 2. Distribuição por período
ax2 = axes[0, 1]
period_order = ['Madrugada', 'Manhã', 'Tarde', 'Noite']
df_model['PERIOD'].value_counts().reindex(period_order).plot(kind='bar', ax=ax2, color=COLORS[1], alpha=0.7)
ax2.set_title('Distribuição por Período', fontsize=12, fontweight='bold')
ax2.set_xlabel('Período')
ax2.set_ylabel('Frequência')
ax2.tick_params(axis='x', rotation=45)

# 3. Fim de semana vs dia de semana
ax3 = axes[0, 2]
weekend_labels = ['Dia de Semana', 'Fim de Semana']
df_model['IS_WEEKEND'].value_counts().sort_index().plot(kind='bar', ax=ax3, color=COLORS[2], alpha=0.7)
ax3.set_xticklabels(weekend_labels, rotation=0)
ax3.set_title('Dia de Semana vs Fim de Semana', fontsize=12, fontweight='bold')
ax3.set_xlabel('')
ax3.set_ylabel('Frequência')

# 4. Distribuição por trimestre
ax4 = axes[1, 0]
df_model['QUARTER'].value_counts().sort_index().plot(kind='bar', ax=ax4, color=COLORS[3], alpha=0.7)
ax4.set_title('Distribuição por Trimestre', fontsize=12, fontweight='bold')
ax4.set_xlabel('Trimestre')
ax4.set_ylabel('Frequência')
ax4.tick_params(axis='x', rotation=0)

# 5. Rush hours
ax5 = axes[1, 1]
rush_data = pd.DataFrame({
    'Rush Manhã': [df_model['IS_MORNING_RUSH'].sum()],
    'Rush Tarde': [df_model['IS_EVENING_RUSH'].sum()],
    'Outros': [len(df_model) - df_model['IS_MORNING_RUSH'].sum() - df_model['IS_EVENING_RUSH'].sum()]
})
rush_data.T.plot(kind='bar', ax=ax5, color=COLORS[4], alpha=0.7, legend=False)
ax5.set_title('Voos em Horários de Rush', fontsize=12, fontweight='bold')
ax5.set_xlabel('')
ax5.set_ylabel('Frequência')
ax5.tick_params(axis='x', rotation=45)

# 6. Temporada de férias
ax6 = axes[1, 2]
holiday_labels = ['Regular', 'Férias']
df_model['IS_HOLIDAY_SEASON'].value_counts().sort_index().plot(kind='bar', ax=ax6, color=COLORS[5], alpha=0.7)
ax6.set_xticklabels(holiday_labels, rotation=0)
ax6.set_title('Temporada Regular vs Férias', fontsize=12, fontweight='bold')
ax6.set_xlabel('')
ax6.set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('../data/processed/fig_fe_01_temporal_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura salva: fig_fe_01_temporal_features.png')

---
## 3. Criação de Features de Contexto Operacional

Features baseadas em características operacionais dos voos e estatísticas históricas.

In [ ]:
# =============================================================================
# FEATURES OPERACIONAIS
# =============================================================================

# 1. DISTANCE_CATEGORY - Categoria de distância do voo
def categorize_distance(distance):
    """Categoriza a distância do voo."""
    if distance < 500:
        return 'Curta'
    elif distance < 1500:
        return 'Média'
    else:
        return 'Longa'

df_model['DISTANCE_CATEGORY'] = df_model['DISTANCE'].apply(categorize_distance)
print('✅ DISTANCE_CATEGORY criada (Curta/Média/Longa)')

# 2. IS_LONG_HAUL - Voo de longa distância (>1500 milhas)
df_model['IS_LONG_HAUL'] = (df_model['DISTANCE'] > 1500).astype(int)
print('✅ IS_LONG_HAUL criada (>1500 milhas)')

# 3. TOTAL_TAXI_TIME - Tempo total de taxi (out + in)
df_model['TOTAL_TAXI_TIME'] = df_model['TAXI_OUT'].fillna(0) + df_model['TAXI_IN'].fillna(0)
print('✅ TOTAL_TAXI_TIME criada (TAXI_OUT + TAXI_IN)')

# 4. TOTAL_DELAY - Atraso total (partida + chegada) - para regressão
df_model['TOTAL_DELAY'] = df_model['DEPARTURE_DELAY'].fillna(0) + df_model['ARRIVAL_DELAY'].fillna(0)
print('✅ TOTAL_DELAY criada (DEPARTURE_DELAY + ARRIVAL_DELAY)')

# 5. DELAY_RECOVERY - Recuperação de atraso durante o voo
# Positivo = recuperou tempo, Negativo = perdeu tempo
df_model['DELAY_RECOVERY'] = df_model['DEPARTURE_DELAY'].fillna(0) - df_model['ARRIVAL_DELAY'].fillna(0)
print('✅ DELAY_RECOVERY criada (DEPARTURE_DELAY - ARRIVAL_DELAY)')

# 6. FLIGHT_EFFICIENCY - Eficiência do voo (tempo real vs programado)
# Valores > 1 indicam voo mais lento que o programado
df_model['FLIGHT_EFFICIENCY'] = np.where(
    df_model['SCHEDULED_TIME'] > 0,
    df_model['ELAPSED_TIME'] / df_model['SCHEDULED_TIME'],
    1.0
)
print('✅ FLIGHT_EFFICIENCY criada (ELAPSED_TIME / SCHEDULED_TIME)')

# Visualiza distribuições
print(f'\n📊 Distribuição DISTANCE_CATEGORY:')
print(df_model['DISTANCE_CATEGORY'].value_counts())

In [ ]:
# =============================================================================
# FEATURES ESTATÍSTICAS POR GRUPO
# =============================================================================
# Estas features capturam padrões históricos

# 1. Média de atraso por companhia aérea
airline_delay_stats = df_model.groupby('AIRLINE').agg({
    'DEPARTURE_DELAY': 'mean',
    'ARRIVAL_DELAY': 'mean'
}).reset_index()
airline_delay_stats.columns = ['AIRLINE', 'AIRLINE_AVG_DEP_DELAY', 'AIRLINE_AVG_ARR_DELAY']

df_model = df_model.merge(airline_delay_stats, on='AIRLINE', how='left')
print('✅ AIRLINE_AVG_DEP_DELAY e AIRLINE_AVG_ARR_DELAY criadas')

# 2. Média de atraso por aeroporto de origem
origin_delay_stats = df_model.groupby('ORIGIN_AIRPORT').agg({
    'DEPARTURE_DELAY': 'mean'
}).reset_index()
origin_delay_stats.columns = ['ORIGIN_AIRPORT', 'ORIGIN_AVG_DELAY']

df_model = df_model.merge(origin_delay_stats, on='ORIGIN_AIRPORT', how='left')
print('✅ ORIGIN_AVG_DELAY criada')

# 3. Média de atraso por aeroporto de destino
dest_delay_stats = df_model.groupby('DESTINATION_AIRPORT').agg({
    'ARRIVAL_DELAY': 'mean'
}).reset_index()
dest_delay_stats.columns = ['DESTINATION_AIRPORT', 'DEST_AVG_DELAY']

df_model = df_model.merge(dest_delay_stats, on='DESTINATION_AIRPORT', how='left')
print('✅ DEST_AVG_DELAY criada')

# 4. Média de atraso por rota (origem-destino)
df_model['ROUTE'] = df_model['ORIGIN_AIRPORT'] + '-' + df_model['DESTINATION_AIRPORT']
route_delay_stats = df_model.groupby('ROUTE').agg({
    'ARRIVAL_DELAY': 'mean',
    'FLIGHT_NUMBER': 'count'
}).reset_index()
route_delay_stats.columns = ['ROUTE', 'ROUTE_AVG_DELAY', 'ROUTE_FREQUENCY']

df_model = df_model.merge(route_delay_stats, on='ROUTE', how='left')
print('✅ ROUTE_AVG_DELAY e ROUTE_FREQUENCY criadas')

# 5. Média de atraso por hora do dia
hour_delay_stats = df_model.groupby('HOUR').agg({
    'DEPARTURE_DELAY': 'mean'
}).reset_index()
hour_delay_stats.columns = ['HOUR', 'HOUR_AVG_DELAY']

df_model = df_model.merge(hour_delay_stats, on='HOUR', how='left')
print('✅ HOUR_AVG_DELAY criada')

# 6. Volume de voos por aeroporto (proxy de congestionamento)
airport_volume = df_model.groupby('ORIGIN_AIRPORT').size().reset_index(name='ORIGIN_VOLUME')
df_model = df_model.merge(airport_volume, on='ORIGIN_AIRPORT', how='left')
print('✅ ORIGIN_VOLUME criada')

print(f'\n📊 Novas features estatísticas criadas!')

---
## 4. Encoding de Variáveis Categóricas

Transformamos variáveis categóricas em formatos numéricos adequados para os modelos.

In [ ]:
# =============================================================================
# ENCODING DE VARIÁVEIS CATEGÓRICAS
# =============================================================================

# Dicionário para armazenar encoders
encoders = {}

# 1. Label Encoding para AIRLINE
le_airline = LabelEncoder()
df_model['AIRLINE_ENCODED'] = le_airline.fit_transform(df_model['AIRLINE'].astype(str))
encoders['airline'] = le_airline
print(f'✅ AIRLINE_ENCODED: {len(le_airline.classes_)} categorias')

# 2. Label Encoding para PERIOD
le_period = LabelEncoder()
df_model['PERIOD_ENCODED'] = le_period.fit_transform(df_model['PERIOD'])
encoders['period'] = le_period
print(f'✅ PERIOD_ENCODED: {len(le_period.classes_)} categorias')

# 3. Label Encoding para DISTANCE_CATEGORY
le_distance = LabelEncoder()
df_model['DISTANCE_CATEGORY_ENCODED'] = le_distance.fit_transform(df_model['DISTANCE_CATEGORY'])
encoders['distance_category'] = le_distance
print(f'✅ DISTANCE_CATEGORY_ENCODED: {len(le_distance.classes_)} categorias')

# 4. Label Encoding para aeroportos (Top N + OTHER)
TOP_N_AIRPORTS = 50

# Origem
top_origin = df_model['ORIGIN_AIRPORT'].value_counts().head(TOP_N_AIRPORTS).index.tolist()
df_model['ORIGIN_AIRPORT_GROUPED'] = df_model['ORIGIN_AIRPORT'].apply(
    lambda x: x if x in top_origin else 'OTHER'
)
le_origin = LabelEncoder()
df_model['ORIGIN_AIRPORT_ENCODED'] = le_origin.fit_transform(df_model['ORIGIN_AIRPORT_GROUPED'])
encoders['origin_airport'] = le_origin
encoders['top_origin_airports'] = top_origin
print(f'✅ ORIGIN_AIRPORT_ENCODED: Top {TOP_N_AIRPORTS} + OTHER')

# Destino
top_dest = df_model['DESTINATION_AIRPORT'].value_counts().head(TOP_N_AIRPORTS).index.tolist()
df_model['DESTINATION_AIRPORT_GROUPED'] = df_model['DESTINATION_AIRPORT'].apply(
    lambda x: x if x in top_dest else 'OTHER'
)
le_dest = LabelEncoder()
df_model['DESTINATION_AIRPORT_ENCODED'] = le_dest.fit_transform(df_model['DESTINATION_AIRPORT_GROUPED'])
encoders['destination_airport'] = le_dest
encoders['top_dest_airports'] = top_dest
print(f'✅ DESTINATION_AIRPORT_ENCODED: Top {TOP_N_AIRPORTS} + OTHER')

# 5. Cyclic encoding para HOUR (captura natureza circular)
df_model['HOUR_SIN'] = np.sin(2 * np.pi * df_model['HOUR'] / 24)
df_model['HOUR_COS'] = np.cos(2 * np.pi * df_model['HOUR'] / 24)
print('✅ HOUR_SIN e HOUR_COS criadas (encoding cíclico)')

# 6. Cyclic encoding para DAY_OF_WEEK
df_model['DOW_SIN'] = np.sin(2 * np.pi * df_model['DAY_OF_WEEK'] / 7)
df_model['DOW_COS'] = np.cos(2 * np.pi * df_model['DAY_OF_WEEK'] / 7)
print('✅ DOW_SIN e DOW_COS criadas (encoding cíclico)')

# 7. Cyclic encoding para MONTH
df_model['MONTH_SIN'] = np.sin(2 * np.pi * df_model['MONTH'] / 12)
df_model['MONTH_COS'] = np.cos(2 * np.pi * df_model['MONTH'] / 12)
print('✅ MONTH_SIN e MONTH_COS criadas (encoding cíclico)')

print(f'\n📊 Total de encoders criados: {len(encoders)}')

---
## 5. Criação das Variáveis Target

Criamos as variáveis target para classificação (IS_DELAYED) e regressão (ARRIVAL_DELAY).

In [ ]:
# =============================================================================
# VARIÁVEIS TARGET
# =============================================================================

# 1. IS_DELAYED - Classificação binária (atraso >= 15 min)
DELAY_THRESHOLD = 15  # Definição FAA de atraso significativo

df_model['IS_DELAYED'] = (df_model['ARRIVAL_DELAY'] >= DELAY_THRESHOLD).astype(int)
print(f'✅ IS_DELAYED criada (threshold = {DELAY_THRESHOLD} min)')
print(f'\n📊 Distribuição IS_DELAYED:')
print(df_model['IS_DELAYED'].value_counts())
print(f'\nProporção de voos atrasados: {df_model["IS_DELAYED"].mean()*100:.2f}%')

# 2. DELAY_SEVERITY - Classificação multiclasse
def categorize_delay_severity(delay):
    """Categoriza a severidade do atraso."""
    if pd.isna(delay):
        return 'Sem Info'
    elif delay <= 0:
        return 'Adiantado/Pontual'
    elif delay < 15:
        return 'Atraso Leve'
    elif delay < 60:
        return 'Atraso Moderado'
    elif delay < 180:
        return 'Atraso Significativo'
    else:
        return 'Atraso Severo'

df_model['DELAY_SEVERITY'] = df_model['ARRIVAL_DELAY'].apply(categorize_delay_severity)
le_severity = LabelEncoder()
df_model['DELAY_SEVERITY_ENCODED'] = le_severity.fit_transform(df_model['DELAY_SEVERITY'])
encoders['delay_severity'] = le_severity
print(f'\n✅ DELAY_SEVERITY criada:')
print(df_model['DELAY_SEVERITY'].value_counts())

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# IS_DELAYED
ax1 = axes[0]
labels = ['Sem Atraso', 'Com Atraso']
colors_pie = ['#2ecc71', '#e74c3c']
df_model['IS_DELAYED'].value_counts().sort_index().plot(
    kind='pie', ax=ax1, labels=labels, colors=colors_pie,
    autopct='%1.1f%%', startangle=90, explode=(0, 0.05)
)
ax1.set_title('Classificação Binária: IS_DELAYED', fontsize=12, fontweight='bold')
ax1.set_ylabel('')

# DELAY_SEVERITY
ax2 = axes[1]
severity_order = ['Adiantado/Pontual', 'Atraso Leve', 'Atraso Moderado', 'Atraso Significativo', 'Atraso Severo']
df_model['DELAY_SEVERITY'].value_counts().reindex(severity_order).plot(
    kind='barh', ax=ax2, color=COLORS
)
ax2.set_title('Classificação Multiclasse: DELAY_SEVERITY', fontsize=12, fontweight='bold')
ax2.set_xlabel('Frequência')

plt.tight_layout()
plt.savefig('../data/processed/fig_fe_02_targets.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Figura salva: fig_fe_02_targets.png')

In [ ]:
# =============================================================================
# ANÁLISE DE DESBALANCEAMENTO DE CLASSES
# =============================================================================

print('📊 ANÁLISE DE DESBALANCEAMENTO DE CLASSES')
print('=' * 50)

# IS_DELAYED
delayed_counts = df_model['IS_DELAYED'].value_counts()
print('\n🎯 IS_DELAYED (Target Binário):')
print(f'   Classe 0 (Sem Atraso): {delayed_counts[0]:,} ({delayed_counts[0]/len(df_model)*100:.2f}%)')
print(f'   Classe 1 (Com Atraso): {delayed_counts[1]:,} ({delayed_counts[1]/len(df_model)*100:.2f}%)')
print(f'   Razão de Desbalanceamento: {delayed_counts[0]/delayed_counts[1]:.2f}:1')

# DELAY_SEVERITY
print('\n🎯 DELAY_SEVERITY (Target Multiclasse):')
severity_counts = df_model['DELAY_SEVERITY'].value_counts()
for cat, count in severity_counts.items():
    print(f'   {cat}: {count:,} ({count/len(df_model)*100:.2f}%)')

# Nota sobre SMOTE
print('\n⚠️ NOTA IMPORTANTE:')
print('   O desbalanceamento será tratado no notebook de classificação (03_modeling_classification.ipynb)')
print('   usando técnicas como SMOTE, class_weight, ou undersampling.')

---
## 6. Análise de Correlação das Novas Features

In [ ]:
# =============================================================================
# ANÁLISE DE CORRELAÇÃO
# =============================================================================

# Seleciona apenas colunas numéricas para correlação
numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()

# Remove colunas irrelevantes ou com muitos NaN
cols_to_exclude = ['YEAR', 'FLIGHT_NUMBER', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME',
                   'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'WHEELS_OFF', 'WHEELS_ON',
                   'CANCELLED', 'DIVERTED']
numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]

# Features importantes para análise de correlação com target
important_features = [
    'IS_DELAYED', 'ARRIVAL_DELAY', 'DEPARTURE_DELAY',
    'DISTANCE', 'SCHEDULED_TIME', 'ELAPSED_TIME',
    'HOUR', 'DAY_OF_WEEK', 'MONTH', 
    'TAXI_OUT', 'TAXI_IN', 'TOTAL_TAXI_TIME',
    'AIRLINE_AVG_DEP_DELAY', 'ORIGIN_AVG_DELAY', 'DEST_AVG_DELAY',
    'HOUR_AVG_DELAY', 'ROUTE_AVG_DELAY',
    'IS_WEEKEND', 'IS_MORNING_RUSH', 'IS_EVENING_RUSH', 'IS_HOLIDAY_SEASON',
    'FLIGHT_EFFICIENCY', 'ORIGIN_VOLUME'
]

# Filtra features que existem
important_features = [f for f in important_features if f in df_model.columns]

print(f'📊 Analisando correlação de {len(important_features)} features')

# Calcula correlação
correlation_matrix = df_model[important_features].corr()

# Correlação com target IS_DELAYED
print('\n🎯 Top 15 Features com maior correlação com IS_DELAYED:')
target_correlation = correlation_matrix['IS_DELAYED'].drop('IS_DELAYED').abs().sort_values(ascending=False)
print(target_correlation.head(15))

In [ ]:
# Heatmap de correlação
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix, 
    mask=mask,
    annot=True, 
    fmt='.2f', 
    cmap='RdBu_r', 
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Matriz de Correlação - Features Importantes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_fe_03_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura salva: fig_fe_03_correlation_matrix.png')

In [ ]:
# Visualização da importância das features (correlação com target)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Correlação com IS_DELAYED (classificação)
ax1 = axes[0]
target_corr_delayed = correlation_matrix['IS_DELAYED'].drop('IS_DELAYED').sort_values()
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in target_corr_delayed]
target_corr_delayed.plot(kind='barh', ax=ax1, color=colors)
ax1.axvline(x=0, color='black', linewidth=0.5)
ax1.set_title('Correlação com IS_DELAYED (Classificação)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Correlação de Pearson')

# 2. Correlação com ARRIVAL_DELAY (regressão)
ax2 = axes[1]
target_corr_arrival = correlation_matrix['ARRIVAL_DELAY'].drop('ARRIVAL_DELAY').sort_values()
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in target_corr_arrival]
target_corr_arrival.plot(kind='barh', ax=ax2, color=colors)
ax2.axvline(x=0, color='black', linewidth=0.5)
ax2.set_title('Correlação com ARRIVAL_DELAY (Regressão)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Correlação de Pearson')

plt.tight_layout()
plt.savefig('../data/processed/fig_fe_04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figura salva: fig_fe_04_feature_importance.png')

---
## 7. Tratamento de Outliers e Valores Faltantes

In [ ]:
# =============================================================================
# TRATAMENTO DE VALORES FALTANTES
# =============================================================================

print('📊 ANÁLISE DE VALORES FALTANTES')
print('=' * 50)

# Conta valores faltantes
missing_counts = df_model.isnull().sum()
missing_pct = (missing_counts / len(df_model) * 100).round(2)
missing_df = pd.DataFrame({
    'Coluna': missing_counts.index,
    'Valores Faltantes': missing_counts.values,
    'Percentual (%)': missing_pct.values
}).sort_values('Valores Faltantes', ascending=False)

# Exibe apenas colunas com valores faltantes
missing_df = missing_df[missing_df['Valores Faltantes'] > 0]
print(f'\n{len(missing_df)} colunas com valores faltantes:')
print(missing_df.to_string(index=False))

In [ ]:
# =============================================================================
# ANÁLISE DE OUTLIERS
# =============================================================================

# Colunas para análise de outliers
outlier_cols = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DISTANCE', 'TAXI_OUT', 'TAXI_IN', 'ELAPSED_TIME']

print('📊 ANÁLISE DE OUTLIERS')
print('=' * 50)

for col in outlier_cols:
    if col in df_model.columns:
        Q1 = df_model[col].quantile(0.25)
        Q3 = df_model[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = ((df_model[col] < lower_bound) | (df_model[col] > upper_bound)).sum()
        outlier_pct = outliers / len(df_model) * 100
        
        print(f'\n{col}:')
        print(f'   Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
        print(f'   Limites: [{lower_bound:.2f}, {upper_bound:.2f}]')
        print(f'   Outliers: {outliers:,} ({outlier_pct:.2f}%)')

In [ ]:
# =============================================================================
# TRATAMENTO DE OUTLIERS EXTREMOS
# =============================================================================
# Nota: Para delays, outliers podem ser importantes! Vamos apenas clippar valores extremos.

# Clipping para valores extremos de delay (>500 min = 8h+)
MAX_DELAY = 500
MIN_DELAY = -60  # Voos podem chegar até 1h adiantado

df_model['DEPARTURE_DELAY_CLIPPED'] = df_model['DEPARTURE_DELAY'].clip(MIN_DELAY, MAX_DELAY)
df_model['ARRIVAL_DELAY_CLIPPED'] = df_model['ARRIVAL_DELAY'].clip(MIN_DELAY, MAX_DELAY)

print(f'✅ Delays clippados para [{MIN_DELAY}, {MAX_DELAY}] minutos')

# Estatísticas após clipping
print(f'\n📊 ARRIVAL_DELAY após clipping:')
print(df_model['ARRIVAL_DELAY_CLIPPED'].describe())

---
## 8. Seleção de Features e Divisão Train/Test

In [ ]:
# =============================================================================
# DEFINIÇÃO DOS CONJUNTOS DE FEATURES
# =============================================================================

# Features para CLASSIFICAÇÃO (prever IS_DELAYED)
# Importante: NÃO incluir features que vazem informação do futuro
features_classification = [
    # Operacionais (disponíveis antes do voo)
    'DISTANCE',
    'SCHEDULED_TIME',
    
    # Temporais
    'HOUR', 'DAY_OF_WEEK', 'MONTH', 'QUARTER',
    'HOUR_SIN', 'HOUR_COS', 'DOW_SIN', 'DOW_COS', 'MONTH_SIN', 'MONTH_COS',
    'IS_WEEKEND', 'IS_MORNING_RUSH', 'IS_EVENING_RUSH', 'IS_HOLIDAY_SEASON',
    'PERIOD_ENCODED',
    
    # Contexto (estatísticas históricas)
    'AIRLINE_ENCODED',
    'ORIGIN_AIRPORT_ENCODED',
    'DESTINATION_AIRPORT_ENCODED',
    'AIRLINE_AVG_DEP_DELAY',
    'AIRLINE_AVG_ARR_DELAY',
    'ORIGIN_AVG_DELAY',
    'DEST_AVG_DELAY',
    'HOUR_AVG_DELAY',
    'ORIGIN_VOLUME',
    
    # Categoria
    'DISTANCE_CATEGORY_ENCODED',
    'IS_LONG_HAUL'
]

# Features para REGRESSÃO (prever ARRIVAL_DELAY)
# Pode incluir DEPARTURE_DELAY pois já ocorreu no momento da previsão
features_regression = features_classification + [
    'DEPARTURE_DELAY_CLIPPED',  # Disponível após decolagem
    'TAXI_OUT'  # Disponível após decolagem
]

# Targets
target_classification = 'IS_DELAYED'
target_regression = 'ARRIVAL_DELAY_CLIPPED'
target_multiclass = 'DELAY_SEVERITY_ENCODED'

print(f'📊 Features para Classificação: {len(features_classification)}')
print(f'📊 Features para Regressão: {len(features_regression)}')

In [ ]:
# =============================================================================
# PREPARAÇÃO DO DATASET FINAL
# =============================================================================

# Verifica quais features existem
features_classification = [f for f in features_classification if f in df_model.columns]
features_regression = [f for f in features_regression if f in df_model.columns]

print(f'✅ Features válidas para Classificação: {len(features_classification)}')
print(f'✅ Features válidas para Regressão: {len(features_regression)}')

# Remove registros com valores faltantes nas features importantes
all_features = list(set(features_classification + features_regression + [target_classification, target_regression]))
df_final = df_model.dropna(subset=[f for f in all_features if f in df_model.columns])

print(f'\n📊 Dataset final: {len(df_final):,} registros')
print(f'   Registros removidos: {len(df_model) - len(df_final):,}')

In [ ]:
# =============================================================================
# DIVISÃO TRAIN/TEST
# =============================================================================

# Separa features e targets
X_clf = df_final[features_classification]
y_clf = df_final[target_classification]

X_reg = df_final[features_regression]
y_reg = df_final[target_regression]

y_multiclass = df_final[target_multiclass]

# Divisão 80/20 com estratificação
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y_clf  # Estratificação para manter proporções
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)

# Divisão para multiclasse
_, _, y_train_multi, y_test_multi = train_test_split(
    X_clf, y_multiclass, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y_multiclass
)

print('✅ Divisão Train/Test realizada (80/20)')
print(f'\n📊 Classificação Binária:')
print(f'   Train: {len(X_train_clf):,} ({len(X_train_clf)/len(X_clf)*100:.1f}%)')
print(f'   Test:  {len(X_test_clf):,} ({len(X_test_clf)/len(X_clf)*100:.1f}%)')
print(f'\n📊 Distribuição de classes no Train:')
print(y_train_clf.value_counts())
print(f'\n📊 Distribuição de classes no Test:')
print(y_test_clf.value_counts())

In [ ]:
# =============================================================================
# SALVAMENTO DOS DADOS
# =============================================================================

# Cria diretório se não existir
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 1. Salva o dataset completo com features
df_final.to_parquet('../data/processed/flights_featured.parquet', index=False)
print('✅ Dataset com features salvo: flights_featured.parquet')

# 2. Salva os dados de treino e teste para classificação
train_clf_df = X_train_clf.copy()
train_clf_df['IS_DELAYED'] = y_train_clf.values
train_clf_df['DELAY_SEVERITY_ENCODED'] = y_train_multi.values
train_clf_df.to_parquet('../data/processed/train_classification.parquet', index=False)
print('✅ Train classificação salvo: train_classification.parquet')

test_clf_df = X_test_clf.copy()
test_clf_df['IS_DELAYED'] = y_test_clf.values
test_clf_df['DELAY_SEVERITY_ENCODED'] = y_test_multi.values
test_clf_df.to_parquet('../data/processed/test_classification.parquet', index=False)
print('✅ Test classificação salvo: test_classification.parquet')

# 3. Salva os dados de treino e teste para regressão
train_reg_df = X_train_reg.copy()
train_reg_df['ARRIVAL_DELAY_CLIPPED'] = y_train_reg.values
train_reg_df.to_parquet('../data/processed/train_regression.parquet', index=False)
print('✅ Train regressão salvo: train_regression.parquet')

test_reg_df = X_test_reg.copy()
test_reg_df['ARRIVAL_DELAY_CLIPPED'] = y_test_reg.values
test_reg_df.to_parquet('../data/processed/test_regression.parquet', index=False)
print('✅ Test regressão salvo: test_regression.parquet')

# 4. Salva os encoders
joblib.dump(encoders, '../models/feature_encoders.pkl')
print('✅ Encoders salvos: feature_encoders.pkl')

# 5. Salva metadados
metadata = {
    'features_classification': features_classification,
    'features_regression': features_regression,
    'target_classification': target_classification,
    'target_regression': target_regression,
    'target_multiclass': target_multiclass,
    'train_size_clf': len(X_train_clf),
    'test_size_clf': len(X_test_clf),
    'train_size_reg': len(X_train_reg),
    'test_size_reg': len(X_test_reg),
    'random_state': RANDOM_STATE,
    'delay_threshold': DELAY_THRESHOLD
}
joblib.dump(metadata, '../models/feature_engineering_metadata.pkl')
print('✅ Metadata salvo: feature_engineering_metadata.pkl')

print('\n' + '=' * 50)
print('✅ FEATURE ENGINEERING CONCLUÍDO!')
print('=' * 50)

In [ ]:
# =============================================================================
# RESUMO FINAL
# =============================================================================

print('📊 RESUMO DO FEATURE ENGINEERING')
print('=' * 60)

print('\n🔹 FEATURES TEMPORAIS CRIADAS:')
temporal_features = ['HOUR', 'MINUTE', 'PERIOD', 'IS_WEEKEND', 'IS_MORNING_RUSH', 
                     'IS_EVENING_RUSH', 'QUARTER', 'IS_HOLIDAY_SEASON',
                     'HOUR_SIN', 'HOUR_COS', 'DOW_SIN', 'DOW_COS', 'MONTH_SIN', 'MONTH_COS']
for f in temporal_features:
    print(f'   ✅ {f}')

print('\n🔹 FEATURES OPERACIONAIS CRIADAS:')
operational_features = ['DISTANCE_CATEGORY', 'IS_LONG_HAUL', 'TOTAL_TAXI_TIME', 
                        'TOTAL_DELAY', 'DELAY_RECOVERY', 'FLIGHT_EFFICIENCY', 'ROUTE']
for f in operational_features:
    print(f'   ✅ {f}')

print('\n🔹 FEATURES ESTATÍSTICAS CRIADAS:')
stat_features = ['AIRLINE_AVG_DEP_DELAY', 'AIRLINE_AVG_ARR_DELAY', 'ORIGIN_AVG_DELAY',
                 'DEST_AVG_DELAY', 'ROUTE_AVG_DELAY', 'ROUTE_FREQUENCY', 'HOUR_AVG_DELAY', 'ORIGIN_VOLUME']
for f in stat_features:
    print(f'   ✅ {f}')

print('\n🔹 ENCODINGS REALIZADOS:')
print(f'   ✅ AIRLINE_ENCODED')
print(f'   ✅ PERIOD_ENCODED')
print(f'   ✅ DISTANCE_CATEGORY_ENCODED')
print(f'   ✅ ORIGIN_AIRPORT_ENCODED (Top 50 + OTHER)')
print(f'   ✅ DESTINATION_AIRPORT_ENCODED (Top 50 + OTHER)')

print('\n🔹 TARGETS CRIADOS:')
print(f'   ✅ IS_DELAYED (binário, threshold={DELAY_THRESHOLD} min)')
print(f'   ✅ DELAY_SEVERITY (multiclasse, 5 categorias)')
print(f'   ✅ ARRIVAL_DELAY_CLIPPED (contínuo para regressão)')

print('\n🔹 ARQUIVOS SALVOS:')
print(f'   📁 data/processed/flights_featured.parquet')
print(f'   📁 data/processed/train_classification.parquet')
print(f'   📁 data/processed/test_classification.parquet')
print(f'   📁 data/processed/train_regression.parquet')
print(f'   📁 data/processed/test_regression.parquet')
print(f'   📁 models/feature_encoders.pkl')
print(f'   📁 models/feature_engineering_metadata.pkl')

print('\n' + '=' * 60)
print('🎯 PRÓXIMO PASSO: Executar notebooks de modelagem')
print('   → 03_modeling_classification.ipynb')
print('   → 04_modeling_regression.ipynb')
print('=' * 60)